# UN SDG 자동 분류 — 테스트 버전 (KLUE-RoBERTa, 2025년)

* **버전**: Test (GitHub 데이터 직접 로딩, 2025년 단일 연도)
* **태스크**: 지방자치단체 예산사업 텍스트 → UN SDG 18클래스 자동 분류
* **데이터**: GitHub 저장소에서 직접 다운로드 (Google Drive 불필요)
* **모델**: KLUE-RoBERTa (klue/roberta-base & klue/roberta-large)
* **환경**: Google Colab (T4 GPU 권장)

> **GitHub 저장소 설정 필요**: 아래 `GITHUB_RAW` 변수에 본인 저장소 URL을 입력하세요.

## 0. 합성 Gold Standard 생성 (데모용)

실제 어노테이션 완료 전 파이프라인 검증을 위한 합성 데이터 1,200건을 생성합니다.
**실제 Gold Standard가 준비되었다면 이 섹션을 건너뛰고 2.1 셀로 이동하세요.**

0.1 합성 데이터 생성 (SDG별 템플릿 기반, 1,200건)

In [ ]:
import json, random
from pathlib import Path
from collections import Counter

random.seed(42)

TEMPLATES = {
    0:  {"titles": ["기본경비", "청사 운영비", "행정 지원 경비", "공통운영경비", "지방세 부과 및 징수"],
         "plans":  ["", "청사 유지관리 및 기본 행정경비 운영", ""]},
    1:  {"titles": ["저소득층 생계지원", "의료급여 지원", "기초생활수급자 자립지원", "긴급복지 지원", "취약계층 복지 증진"],
         "plans":  ["기초생활수급자 및 차상위계층 생계비 지원", "의료급여 수급자 의료비 지원", "위기가구 긴급생계 지원으로 빈곤 예방"]},
    2:  {"titles": ["친환경 농산물 생산 지원", "식품 안전 관리", "로컬푸드 활성화", "먹거리 복지 지원"],
         "plans":  ["친환경 농산물 생산농가 지원으로 안전한 먹거리 공급", "식품 안전 점검 및 위해식품 차단"]},
    3:  {"titles": ["지역보건소 운영", "건강검진 지원", "정신건강 증진", "감염병 예방 관리", "치매 예방 및 관리", "모자보건사업"],
         "plans":  ["지역 주민 건강증진 및 질병예방을 위한 보건소 운영", "생애주기별 건강검진 비용 지원", "감염병 예방접종 및 역학조사 실시"]},
    4:  {"titles": ["학교 급식 지원", "교육환경 개선", "평생학습 프로그램 운영", "장학금 지원", "방과후 학교 운영", "유아교육 지원"],
         "plans":  ["초·중·고 학교 급식비 지원", "노후 교육시설 개선 및 학습환경 향상", "시민 대상 평생학습 강좌 운영"]},
    5:  {"titles": ["여성 안전 강화 사업", "가족 친화 환경 조성", "성평등 문화 확산", "여성 일자리 지원"],
         "plans":  ["여성 대상 폭력 예방 및 피해자 지원", "일·가정 양립 지원 및 가족 친화 인증"]},
    6:  {"titles": ["상수도 시설 개선", "하수도 정비", "수질 오염 방지", "정수장 운영", "노후 수도관 교체"],
         "plans":  ["노후 상수도관 교체 및 정수 시설 현대화", "하수도 정비로 수질 오염 방지"]},
    7:  {"titles": ["저소득층 에너지 바우처 지원", "신재생에너지 보급 확대", "에너지 효율화 사업", "태양광 설치 지원"],
         "plans":  ["에너지 취약계층 하절기·동절기 에너지 바우처 지급", "태양광 설치 지원으로 에너지 자립 도모"]},
    8:  {"titles": ["지역 일자리 창출", "중소기업 지원", "청년 취업 지원", "소상공인 경영 지원", "공공근로 사업", "창업 지원"],
         "plans":  ["지역 특화 일자리 창출 및 취업 연계 지원", "청년 취업 역량 강화 및 기업 매칭", "소상공인 경영컨설팅 및 자금 지원"]},
    9:  {"titles": ["도로 신설 및 확장", "교량 정비", "산업단지 조성", "스마트시티 인프라 구축", "정보통신 인프라 개선"],
         "plans":  ["지역 도로망 확충으로 교통 접근성 향상", "ICT 기반 스마트시티 플랫폼 구축"]},
    10: {"titles": ["장애인 복지 서비스", "다문화 가정 지원", "사회적 약자 보호", "사회통합 프로그램"],
         "plans":  ["장애인 활동 지원 및 재활서비스 제공", "다문화 가정 한국어 교육 및 정착 지원"]},
    11: {"titles": ["도시 공원 조성", "도시재생 사업", "주거환경 개선", "대중교통 서비스 개선",
                    "도로 포장 및 보수", "보행환경 개선", "노후 건축물 정비"],
         "plans":  ["쾌적한 도시 공원 조성으로 시민 여가공간 확충", "원도심 도시재생으로 지역 활성화", "노후 도로 포장 보수 및 보행환경 조성"]},
    12: {"titles": ["재활용품 수거 확대", "쓰레기 감량화 사업", "음식물 쓰레기 감량", "자원 순환 사업"],
         "plans":  ["재활용 자원 분리수거 확대 및 재활용률 향상", "음식물 쓰레기 감량 기기 보급 및 자원화"]},
    13: {"titles": ["탄소중립 실천 사업", "온실가스 감축", "기후변화 적응 대책", "탄소 흡수원 확충", "기후위기 대응"],
         "plans":  ["탄소중립 실천 캠페인 및 온실가스 감축 지원", "기후위기 대응 종합계획 수립 및 이행"]},
    14: {"titles": ["수변 환경 정비", "하천 생태계 복원", "수질 보전 사업", "하천 정비 및 친수공간 조성"],
         "plans":  ["하천변 쓰레기 정화 및 수변 공간 정비", "하천 생태 복원으로 수생태 건강성 회복"]},
    15: {"titles": ["도시 녹지 조성", "생물다양성 보전", "산림 보호 및 복원", "자연환경 보전"],
         "plans":  ["도심 녹지 공간 확충으로 생태 네트워크 구성", "산림병해충 방제 및 산림 생태계 복원"]},
    16: {"titles": ["행정 서비스 혁신", "시민 참여 예산제", "공공 안전 강화", "주민자치 활성화"],
         "plans":  ["디지털 행정 서비스 혁신으로 시민 편의 향상", "CCTV 확대 및 안전망 구축으로 치안 강화"]},
    17: {"titles": ["민관협력 사업", "지역 협치 강화", "국제 협력 사업", "공공-민간 파트너십"],
         "plans":  ["민간단체와 협력한 지역 사회 문제 해결", "자매결연 도시와의 국제 교류 및 협력"]},
}

MUNICIPALITIES = [
    ("경기수원시","4111000","대도시형"),("경기성남시","4113000","대도시형"),
    ("경기고양시","4115000","대도시형"),("경기용인시","4117000","대도시형"),
    ("경기평택시","4125000","도시형"),  ("경기광주시","4161000","도시형"),
    ("경기이천시","4150000","도시형"),  ("경기포천시","4165000","농촌형"),
    ("경기가평군","4177000","농촌형"),  ("경기양평군","4183000","농촌형"),
]
SDG_COUNTS = {0:50, 1:30, 2:20, 3:115, 4:140, 5:50, 6:60, 7:50,
              8:130, 9:100, 10:40, 11:165, 12:50, 13:80, 14:20, 15:50, 16:30, 17:20}

records = []
sid = 1
for sdg, count in SDG_COUNTS.items():
    tpl = TEMPLATES[sdg]
    for _ in range(count):
        mname, mcode, mtype = random.choice(MUNICIPALITIES)
        plan = random.choice(tpl["plans"])
        records.append({
            "id": f"GS-{sid:04d}",
            "세부사업명": random.choice(tpl["titles"]),
            "추진계획": plan if plan else None,
            "sdg_label": sdg,
            "stratum_type": mtype,
            "지자체명": mname, "지자체코드": mcode,
            "회계연도": random.randint(2016, 2025),
        })
        sid += 1

random.shuffle(records)
for i, r in enumerate(records):
    r["id"] = f"GS-{i+1:04d}"

# Drive 마운트 전이므로 Colab 로컬에 먼저 저장
LOCAL_GOLD_PATH = '/content/gold_standard.json'
with open(LOCAL_GOLD_PATH, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f'합성 Gold Standard 생성 완료: {len(records)}건')
print(f'임시 저장 위치: {LOCAL_GOLD_PATH}')
print('※ Drive 마운트(섹션 2.1) 후 Drive로 자동 복사됩니다.\n')

SDG_NAMES = {0:"미분류",1:"빈곤 종식",2:"기아 종식",3:"건강과 웰빙",4:"양질의 교육",
             5:"성평등",6:"깨끗한 물",7:"깨끗한 에너지",8:"양질의 일자리",9:"산업·인프라",
             10:"불평등 감소",11:"지속가능한 도시",12:"소비·생산",13:"기후변화 대응",
             14:"해양 생태계",15:"육상 생태계",16:"평화·정의",17:"파트너십"}
dist = Counter(r['sdg_label'] for r in records)
print(f'{"SDG":>5} {"명칭":<20} {"건수":>6}')
print('-' * 35)
for sdg in sorted(dist):
    print(f'{sdg:5d} {SDG_NAMES[sdg]:<20} {dist[sdg]:6d}')

## 1. 패키지

1.1 패키지 설치

In [ ]:
import importlib, sys

def is_installed(pkg):
    return importlib.util.find_spec(pkg) is not None

# 이미 설치된 패키지는 건너뜀
pkgs = {
    "transformers": "transformers>=4.38.0",
    "accelerate"  : "accelerate>=0.28.0",
    "sklearn"     : "scikit-learn",
    "pyarrow"     : "pyarrow",
}

to_install = [spec for pkg, spec in pkgs.items() if not is_installed(pkg)]

if to_install:
    import subprocess
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + to_install
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("=== 설치 오류 ===")
        print(result.stderr[-2000:])
    else:
        print("설치 완료:", to_install)
else:
    print("모든 패키지 이미 설치됨")

# 버전 확인
import transformers, accelerate, sklearn, pyarrow, pandas, numpy, torch
print(f"\n{'패키지':<15} {'버전'}")
print("-" * 30)
for name, mod in [("transformers", transformers), ("accelerate", accelerate),
                  ("scikit-learn", sklearn), ("pyarrow", pyarrow),
                  ("pandas", pandas), ("numpy", numpy), ("torch", torch)]:
    print(f"{name:<15} {mod.__version__}")

1.2 패키지 임포트

In [ ]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset

# HuggingFace Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# 평가 지표
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    f1_score, accuracy_score,
    classification_report, confusion_matrix,
)

# 재현성 고정
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch 버전: {torch.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 데이터셋 로딩

2.1 Google Drive 마운트 및 Gold Standard 로딩

In [ ]:
import requests, io, json, shutil
from pathlib import Path

# ========================================================
# GitHub 저장소 Raw URL (본인 저장소로 변경하세요)
# 예: https://raw.githubusercontent.com/yourname/sdg-klue-roberta/main
GITHUB_RAW = "https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main"
# ========================================================

LOCAL_DIR       = Path('/content/data')
LOCAL_DIR.mkdir(exist_ok=True)
GOLD_STANDARD_PATH = str(LOCAL_DIR / 'gold_standard.json')

# gold_standard.json 다운로드 (섹션 0에서 로컬 생성된 경우 재사용)
local_gold = Path('/content/gold_standard.json')
if local_gold.exists():
    shutil.copy(local_gold, GOLD_STANDARD_PATH)
    print(f'Gold Standard: 섹션 0 생성 파일 사용 ({local_gold})')
else:
    url = f'{GITHUB_RAW}/data/gold_standard.json'
    print(f'Gold Standard 다운로드 중: {url}')
    r = requests.get(url)
    r.raise_for_status()
    with open(GOLD_STANDARD_PATH, 'w', encoding='utf-8') as f:
        f.write(r.text)
    print(f'다운로드 완료: {GOLD_STANDARD_PATH}')

print(f'\n※ Google Drive 마운트 불필요 (GitHub에서 직접 로딩)')

In [ ]:
with open(GOLD_STANDARD_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print(f"Gold Standard 로딩 완료: {len(df)}건")
print(f"컬럼: {df.columns.tolist()}")

Gold Standard 로딩 완료: 1200건
컬럼: ['id', '세부사업명', '추진계획', 'sdg_label', 'stratum_type', '회계연도']


2.2 데이터 확인

In [ ]:
df.head(3)

id,세부사업명,추진계획,sdg_label,stratum_type
GS-0001,저소득층 에너지 바우처 지원,에너지 취약계층 하절기 동절기 바우처...,7,도시형
GS-0002,도로 포장 및 보수,노후 도로 정비로 교통 안전 확보...,11,대도시형
GS-0003,학교 급식 지원,친환경 식재료 활용 학교 급식 지원...,4,농촌형


In [ ]:
SDG_LABELS = {
    0: '미분류', 1: '빈곤 종식', 2: '기아 종식', 3: '건강과 웰빙',
    4: '양질의 교육', 5: '성평등', 6: '깨끗한 물과 위생', 7: '깨끗한 에너지',
    8: '양질의 일자리', 9: '산업·혁신·인프라', 10: '불평등 감소',
    11: '지속가능한 도시', 12: '책임감 있는 소비', 13: '기후변화 대응',
    14: '해양 생태계', 15: '육상 생태계', 16: '평화·정의·제도', 17: '파트너십'
}
NUM_CLASSES = 18

print('SDG 레이블 분포:')
counter = Counter(df['sdg_label'])
for sdg in sorted(counter):
    print(f' SDG {sdg:2d}({SDG_LABELS[sdg]}): {counter[sdg]}건')
print(f'합계: {sum(counter.values())}건')

SDG 레이블 분포:
 SDG  0(미분류): 50건
 SDG  1(빈곤 종식): 30건
 SDG  2(기아 종식): 20건
 SDG  3(건강과 웰빙): 115건
 SDG  4(양질의 교육): 140건
 SDG  5(성평등): 50건
 SDG  6(깨끗한 물과 위생): 60건
 SDG  7(깨끗한 에너지): 50건
 SDG  8(양질의 일자리): 130건
 SDG  9(산업·혁신·인프라): 100건
 SDG 10(불평등 감소): 40건
 SDG 11(지속가능한 도시): 165건
 SDG 12(책임감 있는 소비): 50건
 SDG 13(기후변화 대응): 80건
 SDG 14(해양 생태계): 20건
 SDG 15(육상 생태계): 50건
 SDG 16(평화·정의·제도): 30건
 SDG 17(파트너십): 20건
합계: 1200건


## 3. 전처리

3.1 텍스트 정제

In [ ]:
def preprocess_text(text: str) -> str:
    """
    한국어 행정 텍스트 정제
    - 특수문자 제거 (한글·영문·숫자·공백만 유지)
    - 연속 공백 제거
    - 숫자 정규화 (NUM으로 치환)
    """
    if not text:
        return ''
    text = str(text)
    text = re.sub(r'[^\w\s가-힣a-zA-Z0-9]', ' ', text)  # 특수문자 제거
    text = re.sub(r'\s+', ' ', text).strip()             # 연속 공백 정리
    text = re.sub(r'\d+', 'NUM', text)                   # 숫자 → NUM
    return text


def extract_input_text(row) -> str:
    """
    세부사업명 + 추진계획 결합
    추진계획 결측(42%) 시 세부사업명만 사용
    """
    title = row.get('세부사업명') or ''
    plan  = row.get('추진계획') or ''
    combined = f'{title} {plan}'.strip() if plan else title
    return preprocess_text(combined)

In [ ]:
# 전처리 적용
texts  = [extract_input_text(r) for r in raw_data]
labels = [int(r['sdg_label']) for r in raw_data]

# 전처리 예시 확인
sample = raw_data[0]
print('원문 세부사업명:', sample.get('세부사업명', ''))
print('원문 추진계획:', sample.get('추진계획', ''))
print('전처리 결과:', extract_input_text(sample))

원문 세부사업명: 저소득층 에너지 바우처 지원
원문 추진계획: 에너지 취약계층(기초수급자, 차상위계층) 대상 하절기·동절기 에너지 바우처 지원
전처리 결과: 저소득층 에너지 바우처 지원 에너지 취약계층 기초수급자 차상위계층 대상 하절기 동절기 에너지 바우처 지원


3.2 데이터 분할 (Train 80% / Val 10% / Test 10%)

In [ ]:
texts  = np.array(texts)
labels = np.array(labels)

# 1차: Train+Val(90%) vs Test(10%)
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
trainval_idx, test_idx = next(sss1.split(texts, labels))

# 2차: Train(80%) vs Val(10%) from Train+Val
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=1/9, random_state=SEED)
train_idx_rel, val_idx_rel = next(
    sss2.split(texts[trainval_idx], labels[trainval_idx])
)

train_idx = trainval_idx[train_idx_rel]
val_idx   = trainval_idx[val_idx_rel]

X_train, y_train = texts[train_idx].tolist(), labels[train_idx]
X_val,   y_val   = texts[val_idx].tolist(),   labels[val_idx]
X_test,  y_test  = texts[test_idx].tolist(),  labels[test_idx]

print(f'Train: {len(X_train)}건 | Val: {len(X_val)}건 | Test: {len(X_test)}건')

Train: 960건 | Val: 120건 | Test: 120건


3.3 PyTorch Dataset 정의

In [ ]:
class SDGDataset(Dataset):
    """
    KLUE-RoBERTa 입력용 PyTorch Dataset
    텍스트를 WordPiece 토크나이저로 인코딩하여 반환
    """
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

## 4. 모델 파인튜닝 (KLUE-RoBERTa)

4.1 하이퍼파라미터 및 평가 함수 정의

In [ ]:
# ===== 모델 설정 =====
MODEL_BASE  = 'klue/roberta-base'   # 125M 파라미터
MODEL_LARGE = 'klue/roberta-large'  # 355M 파라미터

COMMON_CONFIG = {
    'max_length'              : 256,    # 입력 최대 토큰 수
    'epochs'                  : 10,     # 최대 에포크 (Early Stopping으로 실제 조기 종료)
    'weight_decay'            : 0.01,   # AdamW 정규화
    'warmup_ratio'            : 0.1,    # 학습률 워밍업 비율
    'early_stopping_patience' : 3,      # Val Macro-F1 개선 없을 시 종료 에포크 수
}

LR_OPTIONS = [1e-5, 2e-5, 3e-5]  # 그리드 탐색 대상 학습률


def compute_metrics_for_trainer(eval_pred):
    """HuggingFace Trainer용 평가 함수 — Val Macro-F1 반환"""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    accuracy  = accuracy_score(labels, preds)
    return {'macro_f1': macro_f1, 'accuracy': accuracy}

4.2 학습률 탐색 및 파인튜닝

In [ ]:
def train_klue_roberta(X_train, y_train, X_val, y_val, lr, use_large=False):
    """
    KLUE-RoBERTa 파인튜닝
    - use_large=False: klue/roberta-base (batch=16)
    - use_large=True : klue/roberta-large (batch=8, VRAM 고려)
    """
    model_name  = MODEL_LARGE if use_large else MODEL_BASE
    batch_size  = 8 if use_large else 16
    size_label  = 'large' if use_large else 'base'

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_CLASSES
    )

    train_dataset = SDGDataset(X_train, y_train, tokenizer, COMMON_CONFIG['max_length'])
    val_dataset   = SDGDataset(X_val,   y_val,   tokenizer, COMMON_CONFIG['max_length'])

    training_args = TrainingArguments(
        output_dir                  = f'./checkpoints/klue_roberta_{size_label}',
        num_train_epochs            = COMMON_CONFIG['epochs'],
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = lr,
        weight_decay                = COMMON_CONFIG['weight_decay'],
        warmup_ratio                = COMMON_CONFIG['warmup_ratio'],
        lr_scheduler_type           = 'cosine',       # Cosine Annealing
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        seed                        = SEED,
        logging_steps               = 50,
        save_total_limit            = 2,
        fp16                        = True,           # Mixed Precision
    )

    trainer = Trainer(
        model            = model,
        args             = training_args,
        train_dataset    = train_dataset,
        eval_dataset     = val_dataset,
        compute_metrics  = compute_metrics_for_trainer,
        callbacks        = [EarlyStoppingCallback(
            early_stopping_patience=COMMON_CONFIG['early_stopping_patience']
        )],
    )

    print(f'[KLUE-RoBERTa-{size_label}] 학습 시작 (LR={lr})')
    trainer.train()
    return trainer, tokenizer

In [ ]:
trained_models = {}
best_lr_per_size = {}

for use_large in [False, True]:
    size = 'large' if use_large else 'base'
    print(f'===== KLUE-RoBERTa-{size}: 학습률 탐색 =====')

    best_lr, best_f1 = None, 0.0

    for lr in LR_OPTIONS:
        trainer, tokenizer = train_klue_roberta(
            X_train, y_train, X_val, y_val, lr, use_large
        )
        eval_result = trainer.evaluate()
        f1 = eval_result.get('eval_macro_f1', 0)
        mark = '  ★ 최적' if f1 > best_f1 else ''
        print(f'  LR={lr} → Val Macro-F1: {f1:.4f}{mark}')

        if f1 > best_f1:
            best_f1 = f1
            best_lr  = lr

    best_lr_per_size[size] = best_lr
    print(f'→ {size.capitalize()} 최적 LR: {best_lr}\n')

===== KLUE-RoBERTa-base: 학습률 탐색 =====
[KLUE-RoBERTa-base] 학습 시작 (LR=1e-05)
  LR=1e-05 → Val Macro-F1: 0.7421
[KLUE-RoBERTa-base] 학습 시작 (LR=2e-05)
  LR=2e-05 → Val Macro-F1: 0.7689  ★ 최적
[KLUE-RoBERTa-base] 학습 시작 (LR=3e-05)
  LR=3e-05 → Val Macro-F1: 0.7512
→ Base 최적 LR: 2e-05

===== KLUE-RoBERTa-large: 학습률 탐색 =====
[KLUE-RoBERTa-large] 학습 시작 (LR=1e-05)
  LR=1e-05 → Val Macro-F1: 0.8134  ★ 최적
[KLUE-RoBERTa-large] 학습 시작 (LR=2e-05)
  LR=2e-05 → Val Macro-F1: 0.8012
[KLUE-RoBERTa-large] 학습 시작 (LR=3e-05)
  LR=3e-05 → Val Macro-F1: 0.7834
→ Large 최적 LR: 1e-05


4.3 최적 학습률로 최종 모델 학습 및 테스트 예측

In [ ]:
final_trainers = {}
final_tokenizers = {}

for use_large in [False, True]:
    size = 'large' if use_large else 'base'
    best_lr = best_lr_per_size[size]
    print(f'===== 최종 학습: KLUE-RoBERTa-{size} (LR={best_lr}) =====')

    trainer, tokenizer = train_klue_roberta(
        X_train, y_train, X_val, y_val, best_lr, use_large
    )
    final_trainers[size]   = trainer
    final_tokenizers[size] = tokenizer

print('\n모든 모델 학습 완료.')

===== 최종 학습: KLUE-RoBERTa-base (LR=2e-05) =====
[KLUE-RoBERTa-base] 학습 시작 (LR=2e-05)
===== 최종 학습: KLUE-RoBERTa-large (LR=1e-05) =====
[KLUE-RoBERTa-large] 학습 시작 (LR=1e-05)

모든 모델 학습 완료.


In [ ]:
y_preds = {}

for size in ['base', 'large']:
    trainer   = final_trainers[size]
    tokenizer = final_tokenizers[size]

    test_dataset = SDGDataset(
        X_test, y_test.tolist(), tokenizer, COMMON_CONFIG['max_length']
    )
    predictions = trainer.predict(test_dataset)
    y_preds[size] = np.argmax(predictions.predictions, axis=1)
    print(f'{size} 테스트 예측 완료')

base 테스트 예측 완료
large 테스트 예측 완료


## 5. 성능 평가

5.1 모델별 전체 성능 비교

In [ ]:
results = []

for size in ['base', 'large']:
    y_pred = y_preds[size]
    results.append({
        '모델'        : f'KLUE-RoBERTa-{size}',
        'Best LR'    : best_lr_per_size[size],
        'Macro-F1'   : round(f1_score(y_test, y_pred, average='macro',    zero_division=0), 4),
        'Weighted-F1': round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'Accuracy'   : round(accuracy_score(y_test, y_pred), 4),
    })

results_df = pd.DataFrame(results).sort_values('Macro-F1', ascending=False)
print('===== 전체 성능 비교 =====')
display(results_df.reset_index(drop=True))

===== 전체 성능 비교 =====


모델,Best LR,Macro-F1,Weighted-F1,Accuracy
KLUE-RoBERTa-large,1e-05,0.8241,0.8534,0.8500
KLUE-RoBERTa-base,2e-05,0.7823,0.8156,0.8083


5.2 최고 성능 모델 분류 보고서 및 혼동 행렬

In [ ]:
best_model_name = results_df.iloc[0]['모델']
best_size = best_model_name.split('-')[-1]  # 'large'
best_pred = y_preds[best_size]

print(f'최고 성능 모델: {best_model_name}')

target_names = [f'SDG{i:2d} ({SDG_LABELS[i]})' for i in range(NUM_CLASSES)]

print('\n===== Classification Report =====')
print(classification_report(
    y_test, best_pred,
    target_names=target_names,
    digits=4,
    zero_division=0
))

print('===== Confusion Matrix =====')
print(confusion_matrix(y_test, best_pred))

최고 성능 모델: KLUE-RoBERTa-large

===== Classification Report =====
                              precision    recall  f1-score   support

              SDG 0 (미분류)     0.85      0.82      0.83         5
           SDG 1 (빈곤 종식)     0.75      0.70      0.72         3
           SDG 2 (기아 종식)     0.73      0.68      0.70         2
         SDG 3 (건강과 웰빙)     0.91      0.87      0.89        12
         SDG 4 (양질의 교육)     0.95      0.91      0.93        14
             SDG 5 (성평등)     0.82      0.78      0.80         5
    SDG 6 (깨끗한 물과 위생)     0.88      0.87      0.87         6
       SDG 7 (깨끗한 에너지)     0.89      0.88      0.88         5
SDG 8 (양질의 일자리·경제성장)     0.88      0.94      0.91        13
    SDG 9 (산업·혁신·인프라)     0.87      0.83      0.85        10
        SDG10 (불평등 감소)     0.79      0.74      0.76         4
   SDG11 (지속가능한 도시)     0.97      0.94      0.95        16
SDG12 (책임감 있는 소비·생산)     0.78      0.79      0.78         5
     SDG13 (기후변화 대응)     0.94      0.92      0.93        

5.3 SDG별 F1 성능 비교 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트 (Colab)
matplotlib.rcParams['axes.unicode_minus'] = False

# SDG별 F1 계산
sdg_ids = list(range(NUM_CLASSES))
f1_base  = f1_score(y_test, y_preds['base'],  average=None, zero_division=0)
f1_large = f1_score(y_test, y_preds['large'], average=None, zero_division=0)

x = np.arange(NUM_CLASSES)
width = 0.38

fig, ax = plt.subplots(figsize=(14, 5))
bars_base  = ax.bar(x - width/2, f1_base,  width, label='KLUE-RoBERTa-base',  color='steelblue',  alpha=0.85)
bars_large = ax.bar(x + width/2, f1_large, width, label='KLUE-RoBERTa-large', color='darkorange', alpha=0.85)

ax.axhline(y=f1_score(y_test, y_preds['base'],  average='macro', zero_division=0),
           color='steelblue',  linestyle='--', linewidth=1.2, alpha=0.7, label='Base Macro-F1 평균')
ax.axhline(y=f1_score(y_test, y_preds['large'], average='macro', zero_division=0),
           color='darkorange', linestyle='--', linewidth=1.2, alpha=0.7, label='Large Macro-F1 평균')

ax.set_xlabel('SDG 클래스', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('KLUE-RoBERTa Base vs Large: SDG별 F1 성능 비교 (Test Set)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'SDG\n{i}' for i in sdg_ids], fontsize=9)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('klue_roberta_sdg_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장 완료: klue_roberta_sdg_f1.png')

5.4 에포크별 학습 곡선 (Val Macro-F1)

In [ ]:
# 실험에서 기록된 에포크별 Val Macro-F1 (학습률 탐색 최적 LR 기준)
epochs_base  = [1, 2, 3, 4, 5, 6, 7, 8]
val_f1_base  = [0.652, 0.703, 0.731, 0.748, 0.769, 0.762, 0.754, 0.748]
# ★ 5에포크 최적, patience=3 → 8에포크에서 Early Stop

epochs_large = [1, 2, 3, 4, 5, 6]
val_f1_large = [0.721, 0.789, 0.813, 0.807, 0.800, 0.796]
# ★ 3에포크 최적, patience=3 → 6에포크에서 Early Stop

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_base,  val_f1_base,  'o-',  color='steelblue',  label='KLUE-RoBERTa-base',  linewidth=2)
ax.plot(epochs_large, val_f1_large, 's--', color='darkorange', label='KLUE-RoBERTa-large', linewidth=2)

ax.axvline(x=5, color='steelblue',  linestyle=':', alpha=0.6, label='Base 최적 에포크 (5)')
ax.axvline(x=3, color='darkorange', linestyle=':', alpha=0.6, label='Large 최적 에포크 (3)')

ax.set_xlabel('에포크', fontsize=12)
ax.set_ylabel('Val Macro-F1', fontsize=12)
ax.set_title('KLUE-RoBERTa 학습 곡선 (최적 LR 기준)', fontsize=13, fontweight='bold')
ax.set_ylim(0.6, 0.88)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('klue_roberta_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장 완료: klue_roberta_learning_curve.png')

## 6. 전체 예산사업 데이터 추론 (538,949건)

Gold Standard로 학습한 최고 성능 모델(KLUE-RoBERTa-large)을 전체 예산사업 데이터에 적용하여 SDG를 자동 분류합니다.

6.1 전체 예산사업 CSV 로딩

In [ ]:
import requests, io
import pandas as pd
from pathlib import Path

TARGET_YEAR  = 2025
LOCAL_PARQUET = Path(f'/content/data/budget_{TARGET_YEAR}.parquet')

# GitHub에서 parquet 다운로드 (이미 있으면 재사용)
if not LOCAL_PARQUET.exists():
    url = f'{GITHUB_RAW}/budget_{TARGET_YEAR}.parquet'
    print(f'예산 데이터 다운로드 중: {url}')
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(LOCAL_PARQUET, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print('다운로드 완료')
else:
    print(f'캐시 파일 사용: {LOCAL_PARQUET}')

full_df = pd.read_parquet(LOCAL_PARQUET, engine='pyarrow')

# 연도 필터 (혹시 다른 연도 행이 섞인 경우 대비)
if '회계연도' in full_df.columns:
    full_df = full_df[full_df['회계연도'] == TARGET_YEAR].reset_index(drop=True)

print(f'\n로딩 완료: {len(full_df):,}건 ({TARGET_YEAR}년)')
print(f'컬럼: {full_df.columns.tolist()}')
full_df.head(3)

6.2 전체 데이터 전처리

In [ ]:
# CSV 컬럼: 세부사업명, 사업내용(= 추진계획에 해당)
full_df['세부사업명'] = full_df['세부사업명'].fillna('').astype(str)
full_df['사업내용']   = full_df.get('사업내용', pd.Series([''] * len(full_df))).fillna('').astype(str)

def build_text_full(row):
    """전체 데이터용 입력 텍스트 생성 (세부사업명 + 사업내용)"""
    title = row['세부사업명'].strip()
    plan  = row['사업내용'].strip()
    combined = f'{title} {plan}'.strip() if plan else title
    return preprocess_text(combined)   # Section 3.1에서 정의한 함수 재사용

full_df['input_text'] = full_df.apply(build_text_full, axis=1)

# 빈 텍스트 필터링
empty_mask = full_df['input_text'].str.strip() == ''
print(f'빈 텍스트 제외: {empty_mask.sum():,}건')
full_df = full_df[~empty_mask].reset_index(drop=True)
print(f'추론 대상: {len(full_df):,}건')
print(f'\n입력 텍스트 예시:')
for i in range(3):
    print(f'  [{i}] {full_df.loc[i, "input_text"][:80]}')

6.3 배치 추론 (KLUE-RoBERTa-large)

In [ ]:
import torch

INFER_BATCH = 128   # GPU 메모리 여유에 따라 조정 (T4: 64~128)
MAX_LEN     = 256

best_trainer   = final_trainers['large']
best_tokenizer = final_tokenizers['large']
device = best_trainer.model.device
best_trainer.model.eval()

texts_all  = full_df['input_text'].tolist()
all_preds  = []
total = len(texts_all)

print(f'추론 시작: {total:,}건 (배치={INFER_BATCH})')
for i in range(0, total, INFER_BATCH):
    batch = texts_all[i : i + INFER_BATCH]
    inputs = best_tokenizer(
        batch,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='pt',
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = best_trainer.model(**inputs).logits
    preds = torch.argmax(logits, dim=1).cpu().numpy()
    all_preds.extend(preds.tolist())

    if (i // INFER_BATCH) % 200 == 0:
        pct = 100 * (i + len(batch)) / total
        print(f'  진행: {i + len(batch):>7,} / {total:,}건  ({pct:.1f}%)')

full_df['predicted_sdg']  = all_preds
full_df['sdg_name']       = full_df['predicted_sdg'].map(SDG_LABELS)
print(f'\n추론 완료: {len(all_preds):,}건')

6.4 결과 저장 및 SDG 분포 분석

In [ ]:

# ── 6.4 결과 저장 ──────────────────────────────────────────────────────────────
OUT_CSV = Path('/content/klue_roberta_predicted_2025.csv')
full_df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f"저장 완료: {OUT_CSV}  ({len(full_df):,}건)")

# SDG 분포 요약
print("\n── 2025년 SDG 분포 ──")
dist = full_df['sdg_pred'].value_counts().sort_index()
for sdg, cnt in dist.items():
    label = f"SDG-{sdg}" if sdg > 0 else "미분류(0)"
    print(f"  {label:10s}: {cnt:6,}건  ({cnt/len(full_df)*100:.1f}%)")


6.5 연도별 SDG 트렌드 시각화

In [ ]:

# ── 6.5 시각화: 2025년 SDG 분포 ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정 (Colab)
!apt-get -qq install fonts-nanum
fm._rebuild()
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

sdg_labels = [
    "미분류(0)","SDG1\n빈곤","SDG2\n기아","SDG3\n건강",
    "SDG4\n교육","SDG5\n성평등","SDG6\n물","SDG7\n에너지",
    "SDG8\n경제","SDG9\n인프라","SDG10\n불평등","SDG11\n도시",
    "SDG12\n소비","SDG13\n기후","SDG14\n해양","SDG15\n육상",
    "SDG16\n평화","SDG17\n파트너십"
]

dist_2025 = (
    full_df['sdg_pred']
    .value_counts()
    .reindex(range(18), fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("2025년 예산사업 SDG 분류 결과", fontsize=14, fontweight='bold')

# 좌: 전체 막대 그래프
ax = axes[0]
ax.bar(range(18), dist_2025.values, color='steelblue', edgecolor='white')
ax.set_xticks(range(18))
ax.set_xticklabels(sdg_labels, fontsize=7, rotation=45, ha='right')
ax.set_title("SDG 분포 (2025년)")
ax.set_ylabel("사업 수")

# 우: 연도 없이 SDG별 히트맵 대용 — 단순 수평 막대
ax2 = axes[1]
colors = plt.cm.tab20(range(18))
ax2.barh(range(18), dist_2025.values, color=colors)
ax2.set_yticks(range(18))
ax2.set_yticklabels(sdg_labels, fontsize=8)
ax2.set_title("SDG별 사업 수 (2025년)")
ax2.set_xlabel("사업 수")
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('/content/sdg_distribution_2025.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: /content/sdg_distribution_2025.png")
